## **Best-Selling Products by Category**

In [0]:
%run "/Workspace/capstone_project/capstone_project_cyntexa/capstone_bundle/src/gold/customer_metrics"

In [0]:
%sql
create or replace view identifier(:catalog).gold.Best_selling_products as
WITH product_sales AS (

    SELECT
        p.product_id,
        p.product_name,
        p.category,
        SUM(s.quantity) AS total_quantity_sold

    FROM identifier(:catalog).silver.sales_clean s

    JOIN identifier(:catalog).silver.products_scd2 p
        ON s.product_id = p.product_id

    WHERE p.is_current = true

    GROUP BY
        p.product_id,
        p.product_name,
        p.category

)

SELECT

    product_id,

    product_name,

    category,

    total_quantity_sold,

    SUM(total_quantity_sold) OVER(

        PARTITION BY category
        ORDER BY total_quantity_sold DESC

    ) AS running_total_quantity

FROM product_sales

ORDER BY
category,
total_quantity_sold DESC;

## **Product Price Changes Over Time (SCD Type 2 History)**

In [0]:
%sql
SELECT

    product_id,

    product_name,

    version,

    round(price, 2) as price,

    effective_date,

    end_date,

    is_current
FROM identifier(:catalog).silver.products_scd2

ORDER BY
product_id,
version;

## **Supplier Performance Metrics**

In [0]:
%sql
create or replace view identifier(:catalog).gold.Supplier_performance as
WITH supplier_performance AS (

    SELECT

        sp.supplier_id,

        sp.supplier_name,

        COUNT(s.sale_id) AS total_orders,

        ROUND(SUM(s.total_amount),2) AS total_revenue

    FROM identifier(:catalog).silver.sales_clean s

    JOIN identifier(:catalog).silver.products_scd2 p
        ON s.product_id = p.product_id

    JOIN identifier(:catalog).silver.suppliers_clean sp
        ON p.supplier_id = sp.supplier_id

    WHERE p.is_current = true

    GROUP BY

        sp.supplier_id,

        sp.supplier_name

)

SELECT

    supplier_id,

    supplier_name,

    total_orders,

    total_revenue,

    PERCENT_RANK() OVER(

        ORDER BY total_revenue DESC

    ) AS supplier_percent_rank

FROM supplier_performance

ORDER BY total_revenue DESC;